# NB06 — deep-learning baselines: EEGNet and 1D-CNN on raw windows (GPU)

EEGNet-8,2 and a compact 1D-CNN on the same raw 10-s windows as P5, the same 25 selection splits, training rule, subject-equal weighting and subject aggregation as the GCN grid, so every comparison is paired. The only GPU notebook of the study: hard cap 1.25 h per model.

In [ ]:
REPO = 'https://github.com/Devguru-codes/Major_project_IIITN.git'
BRANCH = 'feat/implementation'
WORK = '/kaggle/working'
SRC = '/tmp/eegrep-src'   # code checkout stays out of the saved output

import subprocess, sys, os, json, platform, shutil

def sh(cmd, log=None):
    print('$', cmd, flush=True)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    print(r.stdout[-20000:], r.stderr[-5000:], sep='\n', flush=True)
    if log:
        open(log, 'w').write(r.stdout + '\n' + r.stderr)
    if r.returncode != 0:
        raise RuntimeError(f'command failed ({r.returncode}): {cmd}')
    return r.stdout

shutil.rmtree(SRC, ignore_errors=True)
sh(f'git clone --depth 1 -b {BRANCH} {REPO} {SRC}')
SHA = sh(f'git -C {SRC} rev-parse --short HEAD').strip()
sh(f'pip install -q -e "{SRC}[dev]"', log=f'{WORK}/pip_install.log')
PY = f'cd {SRC} && {sys.executable} -m eegrep'
print('git sha', SHA)

In [ ]:
sh(f'cd {SRC} && {sys.executable} -m pytest -q -p no:cacheprovider tests/test_model_train.py tests/test_stats_ablation.py', log=f'{WORK}/pytest_deep.txt')

In [ ]:
import glob
CACHE = [d for d in glob.glob('/kaggle/input/**/cache', recursive=True) if os.path.exists(f'{d}/index.npz')][0]
PART = sorted(glob.glob('/kaggle/input/**/participants.tsv', recursive=True))[0]
FOLDS = f'{SRC}/splits/folds_ds004504.json'
print(CACHE, PART, FOLDS)
import torch
assert torch.cuda.is_available(), 'no GPU attached: refusing to run on CPU'
print(torch.cuda.get_device_name(0))

In [ ]:
sh(f'{PY} grid --cache {CACHE} --folds {FOLDS} --pipelines P5 --edges identity --tag deep_eegnet --set model.arch=eegnet --device cuda --out {WORK}/deep_runs.jsonl --max-hours 1.25', log=f'{WORK}/deep_eegnet.log')

In [ ]:
sh(f'{PY} grid --cache {CACHE} --folds {FOLDS} --pipelines P5 --edges identity --tag deep_cnn1d --set model.arch=cnn1d --device cuda --out {WORK}/deep_runs.jsonl --max-hours 1.25', log=f'{WORK}/deep_cnn1d.log')

In [ ]:
import torch
sh(f'{sys.executable} -m pip freeze', log=f'{WORK}/pip_freeze.txt')
env = {'git_sha': SHA, 'python': platform.python_version(), 'torch': torch.__version__,
       'cuda': torch.cuda.is_available(), 'cpu_count': os.cpu_count(),
       'kaggle_image': os.environ.get('KAGGLE_DOCKER_IMAGE', 'unknown')}
json.dump(env, open(f'{WORK}/environment.json', 'w'), indent=1)
print(env)